In [ ]:
# ==============================================================================
# PROVOCATION BENCHMARK: GRANGER CAUSALITY TEST & COMPARATIVE PLOTS
# EPU (Baker, Bloom & Davis) vs. VIX Tupiniquim (PCA & LASSO)
# ==============================================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.tsa.seasonal import STL
from statsmodels.tsa.stattools import adfuller, grangercausalitytests

print("--- [PROVOCATION BENCHMARK]: COMPLETE ANALYSIS (VIX PCA & LASSO vs EPU) ---")

# ==============================================================================
# 1. HISTORICAL SERIES INGESTION AND ALIGNMENT
# ==============================================================================
# Load EPU series from Baker, Bloom & Davis database
df_epu = pd.read_excel('epu_baker_bloom_davis_brazil.xlsx', sheet_name='Brazil EPU Index')
df_epu['Data'] = pd.to_datetime(df_epu['year'].astype(str) + '-' + df_epu['month'].astype(str) + '-01')
df_epu = df_epu.rename(columns={'Brazil News-Based EPU': 'EPU'})

# Load both VIX series (PCA & LASSO) generated in the main pipeline
df_vix = pd.read_excel('vix_tupiniquim_historical_series.xlsx') 
df_vix['Data'] = pd.to_datetime(df_vix['Date'])

# Structural Decomposition (STL) for EPU Filtering
print("Applying STL Structural Decomposition (period=13) to EPU series...")
stl_epu = STL(df_epu['EPU'], period=13, robust=True).fit()
df_epu['EPU_SA'] = stl_epu.trend + stl_epu.resid

# Merge and Sample Alignment
df_analise = pd.merge(df_vix, df_epu[['Data', 'EPU_SA']], on='Data', how='inner').sort_values('Data').reset_index(drop=True)
print(f"[ALIGNMENT]: Final common sample with {len(df_analise)} synchronized months.")

# First Difference of EPU_SA (due to unit root in level)
df_analise['d_EPU_SA'] = df_analise['EPU_SA'].diff()
df_analise_clean = df_analise.dropna().copy()

# Standardization of all series to Base Mean = 100 within the sample
df_analise['VIX_PCA_100'] = (df_analise['VIX_PCA'] / df_analise['VIX_PCA'].mean()) * 100
df_analise['VIX_LASSO_100'] = (df_analise['VIX_LASSO'] / df_analise['VIX_LASSO'].mean()) * 100
df_analise['EPU_Base100'] = (df_analise['EPU_SA'] / df_analise['EPU_SA'].mean()) * 100

# ==============================================================================
# 2. STATIONARITY VERIFICATION (ADF TEST)
# ==============================================================================
print("\n--- AUGMENTED DICKEY-FULLER (ADF) TEST ---")
p_vix_pca = adfuller(df_analise['VIX_PCA'])[1]
p_vix_lasso = adfuller(df_analise['VIX_LASSO'])[1]
p_epu = adfuller(df_analise['EPU_SA'])[1]
p_depu = adfuller(df_analise_clean['d_EPU_SA'])[1]

print(f"-> P-value VIX Tupiniquim (PCA - Level): {p_vix_pca:.4f}")
print(f"-> P-value VIX Tupiniquim (LASSO - Level): {p_vix_lasso:.4f}")
print(f"-> P-value EPU_SA (Level): {p_epu:.4f}")
print(f"-> P-value EPU_SA (First Difference): {p_depu:.4e} (Stationary)")

# ==============================================================================
# 3. GRANGER CAUSALITY TEST (Lags 1 to 3)
# ==============================================================================
max_lags = 3
print(f"\n--- RUNNING GRANGER CAUSALITY TEST (Lags 1 to {max_lags}) ---")

# --- 3.1. VIX (PCA) vs EPU ---
print("\n[APPROACH 1 - PCA]:")
print("1.1. VIX Tupiniquim (PCA) -> EPU (News-Based Media)")
gc_pca_to_epu = grangercausalitytests(df_analise_clean[['d_EPU_SA', 'VIX_PCA']], maxlag=max_lags, verbose=False)
for lag in range(1, max_lags + 1):
    p_val = gc_pca_to_epu[lag][0]['ssr_ftest'][1]
    f_stat = gc_pca_to_epu[lag][0]['ssr_ftest'][0]
    status = "REJECT H0 (Granger-causes!)" if p_val < 0.05 else "Fail to Reject H0"
    print(f"   -> Lag {lag}: F-stat = {f_stat:.4f} | P-value = {p_val:.4f} | {status}")

print("\n1.2. EPU (News-Based Media) -> VIX Tupiniquim (PCA)")
gc_epu_to_pca = grangercausalitytests(df_analise_clean[['VIX_PCA', 'd_EPU_SA']], maxlag=max_lags, verbose=False)
for lag in range(1, max_lags + 1):
    p_val = gc_epu_to_pca[lag][0]['ssr_ftest'][1]
    f_stat = gc_epu_to_pca[lag][0]['ssr_ftest'][0]
    status = "REJECT H0 (Granger-causes!)" if p_val < 0.05 else "Fail to Reject H0"
    print(f"   -> Lag {lag}: F-stat = {f_stat:.4f} | P-value = {p_val:.4f} | {status}")

# --- 3.2. VIX (LASSO) vs EPU ---
print("\n[APPROACH 2 - LASSO]:")
print("2.1. VIX Tupiniquim (LASSO) -> EPU (News-Based Media)")
gc_lasso_to_epu = grangercausalitytests(df_analise_clean[['d_EPU_SA', 'VIX_LASSO']], maxlag=max_lags, verbose=False)
for lag in range(1, max_lags + 1):
    p_val = gc_lasso_to_epu[lag][0]['ssr_ftest'][1]
    f_stat = gc_lasso_to_epu[lag][0]['ssr_ftest'][0]
    status = "REJECT H0 (Granger-causes!)" if p_val < 0.05 else "Fail to Reject H0"
    print(f"   -> Lag {lag}: F-stat = {f_stat:.4f} | P-value = {p_val:.4f} | {status}")

print("\n2.2. EPU (News-Based Media) -> VIX Tupiniquim (LASSO)")
gc_epu_to_lasso = grangercausalitytests(df_analise_clean[['VIX_LASSO', 'd_EPU_SA']], maxlag=max_lags, verbose=False)
for lag in range(1, max_lags + 1):
    p_val = gc_epu_to_lasso[lag][0]['ssr_ftest'][1]
    f_stat = gc_epu_to_lasso[lag][0]['ssr_ftest'][0]
    status = "REJECT H0 (Granger-causes!)" if p_val < 0.05 else "Fail to Reject H0"
    print(f"   -> Lag {lag}: F-stat = {f_stat:.4f} | P-value = {p_val:.4f} | {status}")

# ==============================================================================
# 4. ISOLATED DUAL-AXIS PLOTS (VIA NEGATIVA: NO LEGEND)
# ==============================================================================

# --- 4.1. PLOT 1: VIX (PCA) vs EPU (DUAL-AXIS) ---
fig, ax1 = plt.subplots(figsize=(12, 4.5))

# Primary Y-Axis (Left): VIX (PCA)
color1 = 'navy'
ax1.set_xlabel('Years', fontsize=10, fontweight='bold')
ax1.set_ylabel('VIX Tupiniquim via PCA (Base Mean = 100)', color=color1, fontsize=10, fontweight='bold')
ax1.plot(df_analise['Data'], df_analise['VIX_PCA_100'], color=color1, linewidth=2.0)
ax1.tick_params(axis='y', labelcolor=color1)
ax1.axhline(100, color='black', linestyle=':', linewidth=0.8, alpha=0.7)
ax1.grid(True, linestyle=':', alpha=0.6)

# Secondary Y-Axis (Right): EPU
ax2 = ax1.twinx()
color2 = 'darkorange'
ax2.set_ylabel('Brazil EPU Index (Base Mean = 100)', color=color2, fontsize=10, fontweight='bold', rotation=270, labelpad=18)
ax2.plot(df_analise['Data'], df_analise['EPU_Base100'], color=color2, linewidth=1.5, linestyle='--')
ax2.tick_params(axis='y', labelcolor=color2)

plt.tight_layout()
plt.savefig('vix_pca_vs_epu_en.png', dpi=300, bbox_inches='tight')
plt.show()
print("\n[IMAGE SAVED]: 'vix_pca_vs_epu_en.png' (No legend, right axis rotated) generated successfully!")


# --- 4.2. PLOT 2: VIX (LASSO) vs EPU (DUAL-AXIS) ---
fig, ax1 = plt.subplots(figsize=(12, 4.5))

# Primary Y-Axis (Left): VIX (LASSO)
color1 = 'firebrick'
ax1.set_xlabel('Years', fontsize=10, fontweight='bold')
ax1.set_ylabel('VIX Tupiniquim via LASSO (Base Mean = 100)', color=color1, fontsize=10, fontweight='bold')
ax1.plot(df_analise['Data'], df_analise['VIX_LASSO_100'], color=color1, linewidth=2.0)
ax1.tick_params(axis='y', labelcolor=color1)
ax1.axhline(100, color='black', linestyle=':', linewidth=0.8, alpha=0.7)
ax1.grid(True, linestyle=':', alpha=0.6)

# Secondary Y-Axis (Right): EPU
ax2 = ax1.twinx()
color2 = 'darkorange'
ax2.set_ylabel('Brazil EPU Index (Base Mean = 100)', color=color2, fontsize=10, fontweight='bold', rotation=270, labelpad=18)
ax2.plot(df_analise['Data'], df_analise['EPU_Base100'], color=color2, linewidth=1.5, linestyle='--')
ax2.tick_params(axis='y', labelcolor=color2)

plt.tight_layout()
plt.savefig('vix_lasso_vs_epu_en.png', dpi=300, bbox_inches='tight')
plt.show()
print("[IMAGE SAVED]: 'vix_lasso_vs_epu_en.png' (No legend, right axis rotated) generated successfully!")

print("\n--- ANALYSIS COMPLETED SUCCESSFULLY! ---")